In [ ]:
def read_cred():
    f = open("../../../cred.txt", "r")
    cred = f.read()
    f.close()
    return cred

def get_OI(url,start,end,interval='PT1M',tag='xx',auth='xx',hS='00',hF='23'):
	url_all =url+'data-reference='+tag+'&aggregation=TIME'+'&aggregation-function=MEAN'+"&from="+start+"T"+hS+"%3A00%3A00.000Z&to="+end+"T"+hF+"%3A59%3A59.000Z&aggregation-period="+interval
	#print(url_all,auth)
	d_data = pd.read_json(url_all,storage_options={ 'Authorization': 'basic '+ auth})
	# print(d_data['values'][0])
	arr = np.asarray(np.asarray(d_data['values'])[0])
	return d_data['values'][0]

def get_data(tags,start, end):
    liste = list(range(0))
    for tag in tags:
        urlTag = quote(tag, safe=':/?#[]@!$&\'()*+,;=')
        data = get_OI(urlBase,start,end,'PT20M',urlTag,credentials,'00','23')
        df = pd.DataFrame(data)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.set_index('timestamp')
        df = df.rename(columns={'value':tag})
        liste.append(df)
    return liste

def merge_data(liste):
    df = reduce(lambda left,right : pd.merge(left, right,left_index=True,right_index=True,how='outer'),liste)
    return df

def filtering(df):
    df = df.dropna(how="all")
    df = df[(df['A1000M_Poids']>700) & (df['A1000M_Poids']<1300)]
    df = df[(df['A1000M_UV_Auto']>800_000) & (df['A1000M_UV_Auto']<1_300_000)]
    df = df[(df['A1000M_Labo']>700_000) & (df['A1000M_Labo']<1_300_000)]
#    df = df.dropna(how="all", subset=['A1000M_Poids','A1000R_Poids'])
    df = df.dropna(how="all", subset=['A1000M_Poids'])
    return df

def ajoute_cumul(df,col0,col1,ratio,nom):
    """
    Ajoute colonnes au dataframe :
    - nom : (colonne 0 * colonne 1) / ratio
    
    Parameters:
    df : DataFrame pandas
    
    Returns:
    DataFrame avec les nouvelles colonnes
    """
    df[nom] = (df.iloc[:, col0] * df.iloc[:, col1]) / ratio
    
    return df

def ajouter_moyennes_glissantes(df,col0,col1,nom, window=10):
    """
    Ajoute colonnes de moyennes pondérées glissantes :
    - nom : moyenne pondérée de la colonne 2 par la colonne 1 sur les 10 dernières valeurs
    
    Parameters:
    df : DataFrame pandas
    window : int, nombre de valeurs pour la fenêtre glissante (défaut: 10)
    
    Returns:
    DataFrame avec les nouvelles colonnes
    """
    # Extraction des colonnes
    poids = df.iloc[:, col0]
    valeurs = df.iloc[:, col1]

    # Calcul de la moyenne pondérée glissante pour UV
    numerateur = (poids * valeurs).rolling(window=window).sum()
    denominateur = poids.rolling(window=window).sum()
    df[nom] = numerateur / denominateur
        
    return df


urlBase = 'https://www.myserver.com/query?'
credentials = read_cred()
tags_other = ['container','UV','AC','Numéro']
tags_selected = ['Poids','Auto','Labo']
tags = tags_other+tags_selected
start = '2010-01-01'
end = '2025-12-16'
df_list = get_data(tags,start,end)
data = merge_data(df_list)

